Partie 1 : Charger les documents et exécuter le modèle de réorganisation


1. Installez les bibliothèques Pinecone

In [20]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

Que faire : Exécutez cette commande dans votre cellule de notebook (notez le ! pour l’exécution dans un notebook).
Pourquoi : Vous aurez besoin du package client pour interagir avec l’API de Pinecone et de l’utilitaire de notebook pour simplifier l’authentification dans des environnements comme Colab.
💡 Conseil : En cas de conflit de versions, redémarrez votre environnement d'exécution après l'installation.

2. Authentifiez-vous avec une pomme de pin

In [21]:
import os
if not os.environ.get("pcsk_6Wme2U_TQvYwdTJUNN1pjszQkrWYihq6HAFayybASm18RzAyz6gTbK6Uepf4S9qPEecRh1"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

Procédure : Exécutez ce bloc de code exactement comme indiqué. Le système vous demandera votre clé API si elle n’est pas déjà définie.
Pourquoi : Fournir votre clé API de manière sécurisée permet au client de se connecter à votre projet Pinecone sans avoir à coder en dur des secrets dans votre script.
💡 Astuce : Récupérez votre clé API dans votre tableau de bord Pinecone, sous « Clés API ». Gardez-la secrète !

3. Instancier le client Pinecone

In [22]:
from pinecone import Pinecone
api_key = os.environ.get("pcsk_6Wme2U_TQvYwdTJUNN1pjszQkrWYihq6HAFayybASm18RzAyz6gTbK6Uepf4S9qPEecRh1")
pc = Pinecone(api_key=api_key)

Que faire : Copiez ce code à l’identique ; inutile de compléter les points de suspension ! Le client Pinecone moderne détecte automatiquement votre environnement.
Pourquoi : Le client (pc) est votre point d'entrée pour toutes les opérations Pinecone : création d'index, requêtes et réorganisation.
💡 Conseil : Si vous rencontrez des erreurs d'authentification, assurez-vous que votre clé API est correcte et active.

4. Définissez votre requête et vos documents

In [23]:
query = "Tell me about Apple's products"
documents = [
    "La pomme est un fruit savoureux et croquant, souvent cultivé dans les vergers. Elle est riche en fibres et en vitamines.", # Document sur la pomme (fruit)
    "Apple Inc. est une entreprise technologique multinationale qui conçoit, développe et vend des produits électroniques grand public, des logiciels et des services en ligne.", # Document sur Apple (entreprise)
    "Le verger familial produit de nombreuses variétés de pommes, dont la Gala et la Granny Smith, très appréciées pour leur goût sucré ou acidulé.", # Autre document sur la pomme (fruit)
    "L'iPhone est l'un des produits phares d'Apple, réputé pour son design élégant, son système d'exploitation iOS intuitif et ses performances puissantes.", # Autre document sur Apple (entreprise)
    "Les produits Apple, comme l'iPad et le MacBook, sont largement utilisés par les professionnels de la création et les étudiants du monde entier." # Un autre document sur Apple (entreprise)
]

Que faire : Remplacer chacun ...par de véritables documents textuels qui mélangent des références à Apple (entreprise) et à la pomme (fruit).
Pourquoi : Vous avez besoin d’un petit ensemble de documents pour tester la capacité du système de reclassement à distinguer différents contextes d’un même mot.
💡 Conseil : Créez-en certains sur le thème « Apple est un fruit » et d'autres sur le thème « Apple fabrique des iPhones » - cela permet de tester
la compréhension contextuelle.


5. Appelez le service de réévaluation.

In [24]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3 # e.g., 3
)

Que faire : Indiquez top_nle nombre de résultats principaux que vous souhaitez voir apparaître (par exemple, 3).
Pourquoi : top_n cela limite le nombre de résultats réorganisés, afin de ne récupérer que les documents les plus pertinents.
💡 Astuce : Essayez différentes top_nvaleurs pour voir comment le classement change !


6. Examiner les résultats reclassés

In [42]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    if matches is None:
        print("No matches returned from reranking. The 'reranked.data' object is None.")
        return
    for i, m in enumerate(matches):
        print(f"  {i+1}. Score: {m.score:.4f}, Document: {m.document.text}")

show_reranked_results(query, reranked.data) # Now using reranked.data

Query: Tell me about Apple's products
  1. Score: 0.8282, Document: Les produits Apple, comme l'iPad et le MacBook, sont largement utilisés par les professionnels de la création et les étudiants du monde entier.
  2. Score: 0.6871, Document: Apple Inc. est une entreprise technologique multinationale qui conçoit, développe et vend des produits électroniques grand public, des logiciels et des services en ligne.
  3. Score: 0.4215, Document: L'iPhone est l'un des produits phares d'Apple, réputé pour son design élégant, son système d'exploitation iOS intuitif et ses performances puissantes.


In [41]:
print(dir(reranked))
print(reranked)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'rerank_result']
RerankResult(
  model='bge-reranker-v2-m3',
  data=[{
    index=4,
    score=0.82824534,
    document={
        id='4',
        text="Les produits Apple, comme l'iPad et le MacBook, sont largement utilisés par les professionnels de la création et les étudiants du monde entier."
    }
  },{
    index=1,
    score=0.68710536,
    document={
        id='1',
        text='Apple Inc. est une entreprise technologique multinationale qui conçoit, développe et vend des produits électroniques grand public, des logiciels et des services en ligne.'
    }
  },{
    index=3,
    score=0.4214923,
    document={
   

Que faire : Remplacez ...le code par celui qui affiche le rang (i+1), le score de similarité m.scoreet le texte du document m.document.text. Renseignez également l’attribut correct pour reranked.
Pourquoi : L’affichage de ces valeurs démontre comment le système de réévaluation classe les documents et quels scores il leur attribue.
💡 Astuce : Examinez la structure de l’objet réorganisé. Vérifiez s’il possède .dataun .matchesattribut. Plus le score est élevé, plus l’objet est pertinent !

Partie 2 : Mise en place d’un index sans serveur pour les notes médicales


1. Installez les bibliothèques de données et de modèles

In [9]:
!pip install pandas torch transformers

Procédure : Exécutez cette commande d'installation dans une cellule de notebook.
Pourquoi : Vous utiliserez ces bibliothèques pour charger, intégrer et manipuler des données de notes médicales.
💡 Conseil : Cela peut prendre quelques minutes. Prenez un café !


2. Importer les modules et définir les paramètres d'environnement

In [26]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index' # Give your index a name

Que faire : Renseignez le fournisseur de cloud (par exemple « aws »), la région (par exemple « us-east-1 ») et choisissez un nom d’index.
Pourquoi : Vous configurez un index sans serveur adapté à vos besoins en ressources et connectez le client à la région cloud appropriée.
💡 Astuce : La plupart des comptes Pinecone utilisent « aws » et « us-east-1 ». Pour le nom de l’index, essayez quelque chose commemedical-notes-index : .


3. Créer ou recréer l'index

In [37]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384, # This matches our embedding model size (e.g., for 'sentence-transformers/all-MiniLM-L6-v2')
    metric='cosine', # Distance metric for similarity
    spec=spec
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-zxz3sr9.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

Que faire : Remplissez la dimension (384 pour notre modèle) et choisissez une métrique (« cosinus » est recommandé).
Pourquoi : La dimension de l’index doit correspondre aux vecteurs d’intégration que vous allez insérer, sinon les mises à jour échoueront.
💡 Indice : Le modèle d’intégration que nous utiliserons produit des vecteurs de dimension 384. La similarité cosinus est particulièrement adaptée à l’intégration de texte.


Partie 3 : Charger les données d’exemple


1. Téléchargez et lisez le fichier JSONL

In [31]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl" # Insert the GitHub raw URL here
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

Procédure : Insérez l’URL GitHub brute :https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl
Pourquoi : Permet de télécharger des exemples de données de notes médicales déjà traitées et intégrées. 💡 Conseil : Veillez à utiliser l’URL GitHub brute, et non l’URL de consultation de fichier classique.
💡 Conseil : Veillez à utiliser l’URL GitHub « brute », et non l’URL de visualisation de fichier habituelle.

2. Prévisualiser le DataFrame

In [34]:
# Show head of the DataFrame
print("Data shape:", df.shape) # Show number of rows and columns
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


Que faire : Renseignez l’attribut pandas approprié pour afficher les dimensions du DataFrame.
Pourquoi : Cela vous assure d'avoir les bonnes colonnes (par exemple, id, values, metadata) avant l'insertion.
💡 Indice : Quel attribut pandas indique les (lignes, colonnes) d’un DataFrame ?


Partie 4 : Mise à jour des données dans l’index


1. Instancier le client d'index et effectuer une mise à jour

In [44]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df) # Pass the DataFrame

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

In [43]:
# Check the index stats to verify its dimension
index_stats = index.describe_index_stats()
print(f"Current index dimension: {index_stats.dimension}")
print(index_stats)

Current index dimension: 384
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


Que faire : Transmettre la variable DataFrame à la fonction upsert.
Pourquoi : Cela transfère toutes vos données intégrées et métadonnées dans Pinecone pour les requêtes ultérieures.
💡 Indice : Dans quelle variable avez-vous stocké les données des notes médicales ?


2. Attendez la disponibilité

In [45]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0 # What should this be?

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

Vector count:  100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

Que faire : Indiquez la condition - à quel nombre le nombre d’éléments du vecteur doit-il être supérieur ?
Pourquoi : Cela garantit que les vecteurs insérés/mis à jour sont entièrement indexés avant toute tentative de requête.
💡 Indice : Nous voulons attendre qu’il y ait au moins quelques vecteurs dans l’index. Quel est le minimum ?


Partie 5 : Fonction de requête et d’intégration


1. Définissez votre fonction d'intégration

In [48]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=0) # Average over sequence length dimension
    return embedding

Que faire : Indiquez la dimension sur laquelle calculer la moyenne (0 ou 1).
Pourquoi : Convertit les requêtes entrantes dans le même espace vectoriel que vos notes indexées.
💡 Indice : Nous voulons faire la moyenne sur la dimension de longueur de séquence pour obtenir un seul vecteur par entrée.


2. Exécuter une requête de recherche sémantique

In [50]:
# Build a query to search
question = "patient with chest pain and shortness of breath" # Ask a medical question
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Que faire : Rédigez une question médicale et choisissez le nombre de résultats à afficher.
Pourquoi : Récupère les notes les plus similaires sémantiquement dans l’index en fonction de votre requête clinique.
💡 Astuce : Essayez des questions comme « douleur thoracique du patient » ou « traitement d’une fracture ». Vous obtiendrez 5 à 10 résultats.


Partie 6 : Afficher et réorganiser les notes cliniques


1. Afficher les premiers résultats de la recherche

In [52]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}') # What field contains the score?
        print(f' Metadata: {match["metadata"]}') # What field contains metadata?
        print('')

show_results(question, sorted_matches)

Question: 'patient with chest pain and shortness of breath'

Results:
   1. ID: P001
 Score: 0.63860935
 Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P016
 Score: 0.495548159
 Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID: P003
 Score: 0.440167814
 Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   4. ID: P063
 Score: 0.401210278
 Metadata: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

   5. ID: P0100
 Score: 0.400779665
 Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}



Que faire : Renseignez les clés de dictionnaire correctes pour le score et les métadonnées.
Pourquoi : Cela vous permet de voir quelles notes ont été initialement considérées comme les plus pertinentes.
💡 Astuce : Examinez la structure des objets de correspondance dans les résultats de la requête Pinecone.


2. Préparer les documents pour le réexamen

In [54]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

Procédure : Saisissez la clé correcte pour accéder aux métadonnées de chaque match.
Pourquoi : Crée un champ récapitulant les métadonnées de chaque note, que le système de réévaluation pourra utiliser lors du réajustement des scores.
💡 Indice : Quel champ avez-vous utilisé à l’étape précédente pour imprimer les métadonnées ?


3. Exécuter le réordonnancement sans serveur

In [56]:
# Define a more specific query for reranking
refined_query = "patient requiring knee surgery" # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3, # How many top results do you want?
    return_documents=True,
)

Que faire : Créez une requête médicale plus précise et choisissez le nombre de résultats reclassés à renvoyer.
Pourquoi : Le réordonnancement utilise la requête affinée et le champ de métadonnées pour réorganiser les notes en fonction de leurs nouveaux scores de pertinence.
💡 Astuce : Essayez « patient nécessitant une opération du genou » ou « plan de traitement du diabète ». Vous obtiendrez 2 à 3 résultats de recherche parmi les meilleurs.


4. Afficher les résultats reclassés

In [60]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}') # What attribute contains the reranking score?
        print(f' Reranking Field: {match.document.reranking_field}') # What contains the searchable field?
        print('')
show_reranked_results(refined_query, reranked_results.data)

Question: 'patient requiring knee surgery'

Reranked Results:
   1. ID: P0100
 Score: 0.0049245106
 Reranking Field: advice: over-the-counter pain relief, stretching; symptoms: muscle pain

   2. ID: P001
 Score: 0.0019494022
 Reranking Field: symptoms: chest pain; tests: EKG, stress test

   3. ID: P016
 Score: 5.3491327e-05
 Reranking Field: condition: heart murmur; referral: cardiology



Que faire : Renseignez les attributs du score de reclassement, du champ de recherche et de la collection de résultats.
Pourquoi : Cela vous permet de comparer comment le système de reclassement améliore le classement des résultats par rapport à la recherche initiale.
💡 Astuce : Vérifiez la structure de l'objet réorganisé - recherchez .scoreles noms de champs, et .dataou similaires.


5. Nettoyage (facultatif)

In [ ]:
# Delete the index to save resources
pc.delete_index(name=index_name)

Que faire : Exécutez cette commande une fois que vous avez terminé afin d’éviter des frais inutiles.
Pourquoi : Les index sans serveur coûtent cher lorsqu’ils contiennent des données, il faut donc les nettoyer après les tests.
💡 Astuce : Vous pouvez toujours recréer l'index plus tard si nécessaire !